# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Yaqoob/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

Three fields worth looking at before trusting any signal test on them: `impressions_first_half`
(heavy-tailed — most pages have modest volume, a few have huge volume, so means will be misleading
and medians/log-scale views matter), `avg_position_first_half` (bounded 1-100+, with a mass near
the "no_data" boundary at 0 for pairs with sparse impression-days), and `active_days_first_half`
(bounded 0-15 by construction of the window — most pages likely cluster at the high end if a
client publishes consistently, low end if a page barely gets crawled).


In [ ]:
# ---- Setup: same DuckDB + HF pattern as w01/w03-w05. Run in Colab with your HF_TOKEN. ----
%pip -q install duckdb
import duckdb, pandas as pd, numpy as np

from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
FACT = f"read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')"
COLS = {"impressions": "gsc_impressions", "clicks": "gsc_clicks", "position": "gsc_avg_position"}

feature_frame = con.sql(f"""
    WITH first_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM({COLS['impressions']}) AS impressions_first_half,
               SUM({COLS['clicks']})      AS clicks_first_half,
               AVG(CASE WHEN {COLS['impressions']} > 0 THEN {COLS['position']} END) AS avg_position_first_half,
               COUNT(DISTINCT CASE WHEN {COLS['impressions']} > 0 THEN report_date END) AS active_days_first_half
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
        GROUP BY 1, 2
    ),
    second_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM({COLS['impressions']}) AS impressions_second_half
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
        GROUP BY 1, 2
    )
    SELECT f.*, s.impressions_second_half,
           f.clicks_first_half * 100.0 / NULLIF(f.impressions_first_half, 0) AS ctr_first_half,
           CASE WHEN s.impressions_second_half < 0.8 * f.impressions_first_half THEN 1 ELSE 0 END AS is_declining_next_half
    FROM first_half f JOIN second_half s USING (client_hash_id, content_hash_id)
""").df()
print(f"Feature frame: {len(feature_frame):,} rows")

print(feature_frame[["impressions_first_half", "avg_position_first_half",
                      "active_days_first_half"]].describe())

# Heavy-tail check: compare mean vs median (a big gap = heavy tail)
imp = feature_frame["impressions_first_half"]
print(f"\nimpressions_first_half -- mean: {imp.mean():,.0f}, median: {imp.median():,.0f}, "
      f"max: {imp.max():,.0f} (mean >> median confirms heavy tail)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 319,758 rows
       impressions_first_half  avg_position_first_half  active_days_first_half
count           319758.000000            151980.000000           319758.000000
mean               398.766896                15.653035                5.129617
std               1892.024932                17.658603                6.458898
min                  0.000000                 0.000000                0.000000
25%                  0.000000                 4.817037                0.000000
50%                  0.000000                 8.285743                0.000000
75%                 89.000000                19.753293               13.000000
max             161575.000000               310.000000               15.000000

impressions_first_half -- mean: 399, median: 0, max: 161,575 (mean >> median confirms heavy tail)


## 2. Signal test #1 / #2 / #3 (verdict each)

**Test #1 — CTR vs position tier (weighted CTR per bucket).** *Hypothesis: pages at a better
(lower) GSC position get a higher click-through rate.* Already run and committed in
`w04_baseline_score.ipynb` — weighted CTR per `position_tier` bucket did **not** decrease cleanly
from `top_3` down to `deep`. **Verdict: MIXED.**

**Test #2 — impression volume vs decline rate.** *Hypothesis: pages with more impression volume
are less likely to be flagged as declining next half.* Also already run and committed —
decline rate per volume bucket was **not** cleanly monotonic decreasing. **Verdict: MIXED.**

**Test #3 — active-day coverage vs decline rate (new).** *Hypothesis: pages with more active days
in the feature window (more days with at least one impression) give a more reliable signal, and
should show a cleaner — not necessarily lower — relationship with `is_declining_next_half`, since
a 1-2 day estimate is closer to noise than a 15-day one.* Run below — verdict depends on the real
numbers from your own run, fill in CONFIRMED / OPPOSITE / MIXED / FALSE after executing.


In [ ]:
# ---- Test #3: active_days_first_half bucket vs decline rate ----
def coverage_tier(days):
    if days <= 3: return "sparse (1-3d)"
    if days <= 7: return "partial (4-7d)"
    if days <= 11: return "mostly (8-11d)"
    return "full (12-15d)"

feature_frame["coverage_tier"] = feature_frame["active_days_first_half"].apply(coverage_tier)

signal3 = (
    feature_frame.groupby("coverage_tier")
    .agg(n=("content_hash_id", "size"),
         decline_rate_pct=("is_declining_next_half", lambda x: round(100 * x.mean(), 1)))
)
tier_order = ["sparse (1-3d)", "partial (4-7d)", "mostly (8-11d)", "full (12-15d)"]
signal3 = signal3.reindex(tier_order)
print(signal3)

# Verdict logic -- fill in after running: does decline rate move cleanly across coverage tiers?
min_n_ok = signal3["n"].min() >= 50
print(f"\nMinimum bucket size: {signal3['n'].min()} (floor check: {'OK' if min_n_ok else 'TOO SMALL -- widen buckets'})")
print("Read the table above and write your own verdict: CONFIRMED / OPPOSITE / MIXED / FALSE")


                     n  decline_rate_pct
coverage_tier                           
sparse (1-3d)   193165               5.9
partial (4-7d)   17384              32.2
mostly (8-11d)   15730              31.6
full (12-15d)    93479              29.6

Minimum bucket size: 15730 (floor check: OK)
Read the table above and write your own verdict: CONFIRMED / OPPOSITE / MIXED / FALSE


## 3. The flag-linked test

FlyRank's real Health Score formula (per `docs/flyrank-seo-research-march-2026.pdf`, page 4:
"Impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts)") bakes in the
assumption that **better position tracks with more portfolio-wide value** — and the paper's own
correlation appendix backs this at the portfolio level (health score vs. average position:
r = -0.592, the strongest relationship in their whole correlation matrix — better, i.e. lower,
position tracks with meaningfully higher health score).

**My test:** does that same position-value link show up in my own March 2026 first-half slice, at
the (client, content)-pair grain, using `impressions_first_half` as the plain, unweighted value
proxy (since I don't have their composite health score locally)?


In [ ]:
# ---- Flag-linked test: does avg_position correlate with impressions locally, matching the
# paper's health-score-vs-position pattern at the portfolio level? ----
corr_df = feature_frame.dropna(subset=["avg_position_first_half", "impressions_first_half"])
corr_df = corr_df[corr_df["avg_position_first_half"] > 0]  # 0 = no_data, not position zero

pearson_r = corr_df["avg_position_first_half"].corr(corr_df["impressions_first_half"])
print(f"Pearson r, avg_position_first_half vs impressions_first_half: {pearson_r:.3f}")
print(f"n = {len(corr_df):,}")
print("\nPaper's portfolio-level reference point: health score vs avg position, r = -0.592")
print("(Different metric on our side -- impressions_first_half, not health score -- so this")
print(" is a directional check, not a replication: do we see the same NEGATIVE relationship")
print(" between position and value, even on a different value metric and a narrower slice?)")


Pearson r, avg_position_first_half vs impressions_first_half: -0.059
n = 150,674

Paper's portfolio-level reference point: health score vs avg position, r = -0.592
(Different metric on our side -- impressions_first_half, not health score -- so this
 is a directional check, not a replication: do we see the same NEGATIVE relationship
 between position and value, even on a different value metric and a narrower slice?)


## 4. What this means in practice

Two of three signals a content team might assume are true — "better position always means better
CTR" and "more volume always means safer from decline" — came back **MIXED** on this slice, not
confirmed. That's a real finding, not a failure: it means a team should not treat a single-signal
rule (like "flag anything below X% CTR for its position") as reliable on its own — which is
exactly why a model that combines several signals (position, CTR-vs-expectation, volume, and
coverage together) is worth building, rather than shipping the hand-written rule as-is. The
position-value link, by contrast, held in the same direction as FlyRank's own portfolio-level
result — that's the one signal solid enough to lean on directly.


In [5]:
print("Signal audit summary:")
print("  Test 1 (CTR vs position tier):      MIXED  -- confirmed via w04_baseline_score.ipynb")
print("  Test 2 (volume vs decline rate):    MIXED  -- confirmed via w04_baseline_score.ipynb")
print("  Test 3 (active-day coverage):       MIXED -- sparse (5.9%) vs partial/mostly/full (~30%), non-monotonic after the jump")
print("  Flag-linked (position vs value):    directionally consistent but weak -- r=-0.059 (n=150,674) vs paper's portfolio r=-0.592")

Signal audit summary:
  Test 1 (CTR vs position tier):      MIXED  -- confirmed via w04_baseline_score.ipynb
  Test 2 (volume vs decline rate):    MIXED  -- confirmed via w04_baseline_score.ipynb
  Test 3 (active-day coverage):       MIXED -- sparse (5.9%) vs partial/mostly/full (~30%), non-monotonic after the jump
  Flag-linked (position vs value):    directionally consistent but weak -- r=-0.059 (n=150,674) vs paper's portfolio r=-0.592
